# Shop assistant instruction dataset

This notebook builds an instruction dataset so a small LLM (Qwen3 0.6B) learns to turn a customer message into the next shop action as JSON.

The model does **not** memorize the catalog, prices, or stock. Those stay in tools. Fine-tuning teaches format and store policy: when to search, ask, add to cart, track, return, or hand off to a human.

## Preparing the instruction dataset

Each example is a dict with string fields `instruction`, `input`, and `output`. `output` is a compact JSON string with keys `action`, `slots`, `missing`, and `reply`.

Closed action set:

| Action | When |
|---|---|
| `search_products` | Category plus useful filters |
| `ask` | Missing `size`, `max_price`, or `order_id` |
| `add_to_cart` | Customer picks a shown SKU |
| `track_order` | Order id present |
| `start_return` | Return with order id and reason |
| `handoff_human` | Complaint, policy exception, or abuse |

Customer messages and `reply` are mixed Spanish / English. JSON keys stay English.

In [ ]:
INSTRUCTION = (
    "You are a shop assistant for a clothing and shoe store. "
    "Read the customer message and return only the next action as JSON "
    "with keys action, slots, missing, and reply. "
    "Do not invent products, prices, or stock."
)

ACTIONS = {
    "search_products",
    "ask",
    "add_to_cart",
    "track_order",
    "start_return",
    "handoff_human",
}

# Hand-written examples (one ES, one EN) for inspection before generation.
EXAMPLE_ES = {
    "instruction": INSTRUCTION,
    "input": "busco unos tenis pa correr, talla 42, que no sean negros",
    "output": (
        '{"action":"search_products",'
        '"slots":{"category":"running shoes","size":"42","exclude_color":["black"]},'
        '"missing":[],'
        '"reply":"Buscando tenis para correr talla 42, excluyendo negro."}'
    ),
}

EXAMPLE_EN = {
    "instruction": INSTRUCTION,
    "input": (
        "Assistant showed:\n"
        "1) Pace Runner Blue sku=SKU-1042\n"
        "2) Trail Flex Red sku=SKU-2281\n"
        "Customer: the second one, add it"
    ),
    "output": (
        '{"action":"add_to_cart",'
        '"slots":{"sku":"SKU-2281"},'
        '"missing":[],'
        '"reply":"Adding SKU-2281 to your cart."}'
    ),
}

print("ES example:")
print(EXAMPLE_ES)
print("\nEN example:")
print(EXAMPLE_EN)

### Synthetic generator

Template banks plus small catalogs. No LLM calls. `random.Random(42)` fills slots.

`ponytail:` variety is only as good as the template bank. If the model memorizes phrasing instead of reading slots, add paraphrases here—no second generator needed.

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path


def dump_output(action: str, slots: dict, missing: list, reply: str) -> str:
    """Serialize the shop action with a fixed key order."""
    return json.dumps(
        {"action": action, "slots": slots, "missing": missing, "reply": reply},
        ensure_ascii=False,
        separators=(",", ":"),
    )


CATEGORIES = {
    "en": [
        ("running shoes", "running"),
        ("sneakers", "casual"),
        ("hiking boots", "hiking"),
        ("sandals", "summer"),
        ("jeans", "denim"),
        ("hoodie", "streetwear"),
        ("t-shirt", "basic"),
        ("jacket", "outerwear"),
    ],
    "es": [
        ("tenis para correr", "running"),
        ("zapatillas", "casual"),
        ("botas de trekking", "hiking"),
        ("sandalias", "summer"),
        ("jeans", "denim"),
        ("hoodie", "streetwear"),
        ("camiseta", "basic"),
        ("chaqueta", "outerwear"),
    ],
}

# Slot values stay English in JSON; display words follow the message language.
COLORS = {
    "en": [("black", "black"), ("white", "white"), ("blue", "blue"), ("red", "red"), ("green", "green")],
    "es": [("black", "negro"), ("white", "blanco"), ("blue", "azul"), ("red", "rojo"), ("green", "verde")],
}
SIZES = ["38", "39", "40", "41", "42", "43", "44", "S", "M", "L"]
PRICES_EN = [50, 80, 100, 120, 150, 200]
PRICES_ES = [150000, 200000, 250000, 300000, 400000]
DELIVERY_DAYS = [3, 5, 7]
PRODUCT_NAMES = [
    "Pace Runner", "Trail Flex", "City Soft", "Peak Hiker", "Urban Hood", "Denim Straight"
]
RETURN_REASONS = {
    "en": [
        ("wrong_size", "wrong size"),
        ("changed_mind", "changed my mind"),
        ("defective", "it arrived defective"),
    ],
    "es": [
        ("wrong_size", "talla incorrecta"),
        ("changed_mind", "cambié de opinión"),
        ("defective", "llegó defectuoso"),
    ],
}


def _order_id(rng: random.Random) -> str:
    return f"ORD-{rng.randint(10000, 99999)}"


def _sku(rng: random.Random) -> str:
    return f"SKU-{rng.randint(1000, 9999)}"


def make_search(rng: random.Random, lang: str) -> dict:
    cat_label, _ = rng.choice(CATEGORIES[lang])
    cat_slot = cat_label if lang == "en" else {
        "tenis para correr": "running shoes",
        "zapatillas": "sneakers",
        "botas de trekking": "hiking boots",
        "sandalias": "sandals",
        "jeans": "jeans",
        "hoodie": "hoodie",
        "camiseta": "t-shirt",
        "chaqueta": "jacket",
    }[cat_label]
    size = rng.choice(SIZES)
    color_slot, color_word = rng.choice(COLORS[lang])
    exclude = rng.random() < 0.45
    price = rng.choice(PRICES_EN if lang == "en" else PRICES_ES)
    days = rng.choice(DELIVERY_DAYS)

    slots: dict = {"category": cat_slot}
    if rng.random() < 0.85:
        slots["size"] = size
    if exclude:
        slots["exclude_color"] = [color_slot]
    elif rng.random() < 0.5:
        slots["color"] = color_slot
    if rng.random() < 0.55:
        slots["max_price"] = price
    if rng.random() < 0.35:
        slots["max_delivery_days"] = days

    if lang == "en":
        bits = [f"looking for {cat_label}"]
        if "size" in slots:
            bits.append(f"size {size}")
        if "exclude_color" in slots:
            bits.append(f"not {color_word}")
        elif "color" in slots:
            bits.append(f"in {color_word}")
        if "max_price" in slots:
            bits.append(f"under ${price}")
        if "max_delivery_days" in slots:
            bits.append(f"arrive within {days} days")
        templates = [
            "I need {msg}",
            "Can you find {msg}?",
            "Show me {msg}",
            "Hi, I'm shopping for {msg}",
            "Search {msg} please",
            "Got anything like {msg}?",
            "Looking for {msg}, thanks",
            "Help me find {msg}",
        ]
        msg = ", ".join(bits)
        customer = rng.choice(templates).format(msg=msg)
        reply = f"Searching for {cat_label}" + (
            f", size {slots['size']}" if "size" in slots else ""
        ) + "."
    else:
        bits = [f"busco {cat_label}"]
        if "size" in slots:
            bits.append(f"talla {size}")
        if "exclude_color" in slots:
            bits.append(f"que no sean {color_word}")
        elif "color" in slots:
            bits.append(f"color {color_word}")
        if "max_price" in slots:
            bits.append(f"menos de {price}")
        if "max_delivery_days" in slots:
            bits.append(f"que lleguen en {days} días")
        templates = [
            "{msg}",
            "Hola, {msg}",
            "¿Tienen {msg}?",
            "Necesito {msg}",
            "Quiero {msg}, porfa",
            "Me ayudas con {msg}",
            "Buscando {msg}",
            "Oye, {msg}",
        ]
        msg = ", ".join(bits)
        customer = rng.choice(templates).format(msg=msg)
        reply = f"Buscando {cat_label}" + (
            f", talla {slots['size']}" if "size" in slots else ""
        ) + "."

    return {
        "instruction": INSTRUCTION,
        "input": customer,
        "output": dump_output("search_products", slots, [], reply),
    }


def make_ask(rng: random.Random, lang: str) -> dict:
    missing_key = rng.choice(["size", "max_price", "order_id"])
    if missing_key == "order_id":
        if lang == "en":
            customer = rng.choice([
                "Where is my order?",
                "Can you track my package?",
                "I want to check my shipment status",
                "Has my order shipped yet?",
                "Track my order please",
                "Still waiting on my order",
                "Any update on delivery?",
                "I need tracking info",
            ])
            reply = "What is your order number?"
        else:
            customer = rng.choice([
                "¿Dónde está mi pedido?",
                "Quiero rastrear mi envío",
                "¿Ya salió mi compra?",
                "Necesito el estado del pedido",
                "¿Me ayudas a rastrear?",
                "Aún no me llega el paquete",
                "¿Hay novedades del envío?",
                "Quiero el tracking",
            ])
            reply = "¿Cuál es el número de pedido?"
        return {
            "instruction": INSTRUCTION,
            "input": customer,
            "output": dump_output("ask", {}, ["order_id"], reply),
        }

    cat_label, _ = rng.choice(CATEGORIES[lang])
    cat_slot = cat_label if lang == "en" else {
        "tenis para correr": "running shoes",
        "zapatillas": "sneakers",
        "botas de trekking": "hiking boots",
        "sandalias": "sandals",
        "jeans": "jeans",
        "hoodie": "hoodie",
        "camiseta": "t-shirt",
        "chaqueta": "jacket",
    }[cat_label]
    slots = {"category": cat_slot}

    if missing_key == "size":
        if lang == "en":
            customer = rng.choice([
                f"I want {cat_label}",
                f"Show me {cat_label}",
                f"Looking for {cat_label}, not sure about size",
                f"Do you have {cat_label}?",
                f"Need some {cat_label}",
                f"Browse {cat_label} for me",
                f"Any {cat_label} in stock?",
                f"Help me pick {cat_label}",
            ])
            reply = "What size do you need?"
        else:
            customer = rng.choice([
                f"Quiero {cat_label}",
                f"Muéstrame {cat_label}",
                f"Busco {cat_label}, no sé la talla",
                f"¿Tienen {cat_label}?",
                f"Necesito {cat_label}",
                f"Ayúdame a elegir {cat_label}",
                f"¿Hay {cat_label}?",
                f"Quiero ver {cat_label}",
            ])
            reply = "¿Qué talla necesitas?"
        missing = ["size"]
    else:
        size = rng.choice(SIZES)
        slots["size"] = size
        if lang == "en":
            customer = rng.choice([
                f"Looking for {cat_label} size {size}, nothing too expensive",
                f"Need {cat_label} in {size}, what's a good price range?",
                f"{cat_label} size {size}, keep it affordable",
                f"Show {cat_label} size {size} but I'm on a budget",
                f"{cat_label}, size {size}, not sure how much to spend",
                f"Find {cat_label} size {size} please",
                f"I want {cat_label} {size}, cheapish",
                f"Size {size} {cat_label}, cost matters",
            ])
            reply = "What is your maximum budget?"
        else:
            customer = rng.choice([
                f"Busco {cat_label} talla {size}, que no sean caros",
                f"Necesito {cat_label} en {size}, no sé cuánto gastar",
                f"{cat_label} talla {size}, algo económico",
                f"Muéstrame {cat_label} talla {size}, voy corto de plata",
                f"Quiero {cat_label} {size}, precio razonable",
                f"{cat_label} talla {size} porfa",
                f"Buscando {cat_label} {size}, baratitos",
                f"Talla {size} de {cat_label}, me importa el precio",
            ])
            reply = "¿Hasta cuánto quieres gastar?"
        missing = ["max_price"]

    return {
        "instruction": INSTRUCTION,
        "input": customer,
        "output": dump_output("ask", slots, missing, reply),
    }


def make_add_to_cart(rng: random.Random, lang: str) -> dict:
    sku1, sku2 = _sku(rng), _sku(rng)
    while sku2 == sku1:
        sku2 = _sku(rng)
    name1, name2 = rng.sample(PRODUCT_NAMES, 2)
    color1 = rng.choice(COLORS["en"])[0]
    color2 = rng.choice(COLORS["en"])[0]
    pick_second = rng.random() < 0.5
    chosen = sku2 if pick_second else sku1

    listing = (
        f"1) {name1} {color1.title()} sku={sku1}\n"
        f"2) {name2} {color2.title()} sku={sku2}"
    )
    if lang == "en":
        choice = rng.choice(
            ["the second one, add it", "I'll take #2", "add the second"]
            if pick_second
            else ["the first one please", "add #1", "I'll take the first"]
        )
        customer = f"Assistant showed:\n{listing}\nCustomer: {choice}"
        reply = f"Adding {chosen} to your cart."
    else:
        choice = rng.choice(
            ["el segundo, agrégalo", "me llevo el 2", "añade el segundo"]
            if pick_second
            else ["el primero porfa", "añade el 1", "me quedo con el primero"]
        )
        customer = f"Asistente mostró:\n{listing}\nCliente: {choice}"
        reply = f"Agregando {chosen} al carrito."

    return {
        "instruction": INSTRUCTION,
        "input": customer,
        "output": dump_output("add_to_cart", {"sku": chosen}, [], reply),
    }


def make_track_order(rng: random.Random, lang: str) -> dict:
    oid = _order_id(rng)
    if lang == "en":
        customer = rng.choice([
            f"Where is order {oid}?",
            f"Track {oid} please",
            f"Status of my order {oid}?",
            f"Has {oid} shipped?",
            f"I need tracking for {oid}",
            f"Update on {oid}",
            f"Can you check order {oid}?",
            f"Looking up shipment {oid}",
        ])
        reply = f"Looking up order {oid}."
    else:
        customer = rng.choice([
            f"¿Dónde está el pedido {oid}?",
            f"Rastrea {oid} porfa",
            f"Estado de mi pedido {oid}",
            f"¿Ya salió {oid}?",
            f"Necesito tracking de {oid}",
            f"Novedades de {oid}",
            f"¿Puedes revisar el pedido {oid}?",
            f"Busco el envío {oid}",
        ])
        reply = f"Consultando el pedido {oid}."
    return {
        "instruction": INSTRUCTION,
        "input": customer,
        "output": dump_output("track_order", {"order_id": oid}, [], reply),
    }


def make_start_return(rng: random.Random, lang: str) -> dict:
    oid = _order_id(rng)
    reason_key, reason_phrase = rng.choice(RETURN_REASONS[lang])
    if lang == "en":
        customer = rng.choice([
            f"I want to return order {oid}, {reason_phrase}",
            f"Start a return for {oid}: {reason_phrase}",
            f"Order {oid} — {reason_phrase}, please return",
            f"Return {oid} because {reason_phrase}",
            f"Can I send back {oid}? {reason_phrase}",
            f"Need a return on {oid}, {reason_phrase}",
            f"{oid} should be returned, {reason_phrase}",
            f"Please process a return for {oid} ({reason_phrase})",
        ])
        reply = f"Starting a return for order {oid}."
    else:
        customer = rng.choice([
            f"Quiero devolver el pedido {oid}, {reason_phrase}",
            f"Inicia una devolución de {oid}: {reason_phrase}",
            f"Pedido {oid} — {reason_phrase}, por favor devolución",
            f"Devolver {oid} porque {reason_phrase}",
            f"¿Puedo devolver {oid}? {reason_phrase}",
            f"Necesito devolución de {oid}, {reason_phrase}",
            f"{oid} hay que devolverlo, {reason_phrase}",
            f"Procesa la devolución de {oid} ({reason_phrase})",
        ])
        reply = f"Iniciando devolución del pedido {oid}."
    return {
        "instruction": INSTRUCTION,
        "input": customer,
        "output": dump_output(
            "start_return",
            {"order_id": oid, "reason": reason_key},
            [],
            reply,
        ),
    }


def make_handoff(rng: random.Random, lang: str) -> dict:
    kind = rng.choice(["complaint", "policy", "abuse"])
    if lang == "en":
        texts = {
            "complaint": [
                "This is the third delay and I'm furious",
                "My package was damaged and support ignored me",
                "I've been waiting 10 days, this is unacceptable",
                "Your delivery ruined my weekend plans",
                "Nobody answers my tickets, escalate this",
                "Lost package again, I want a manager",
                "This store keeps failing me",
                "I need a human, the bot is useless",
            ],
            "policy": [
                "Give me 40% off or I'm leaving",
                "Can you match a competitor's price below cost?",
                "I want free shipping on a $5 item",
                "Refund me outside the return window",
                "Apply a discount code that expired last year",
                "Ship internationally for free today",
                "Override your return policy for me",
                "I demand a permanent VIP discount",
            ],
            "abuse": [
                "You're all idiots, fix my order now",
                "This shop is a scam, go to hell",
                "Useless trash support",
                "Shut up and give me money",
                "I hate this store and everyone here",
                "Morons lost my package",
                "Stop wasting my time you bots",
                "I'll sue you idiots",
            ],
        }
        reply = "Connecting you with a human agent."
    else:
        texts = {
            "complaint": [
                "Es el tercer retraso y estoy furioso",
                "El paquete llegó dañado y nadie responde",
                "Llevo 10 días esperando, es inaceptable",
                "El envío me arruinó el fin de semana",
                "Nadie contesta mis tickets, escala esto",
                "Otra vez perdieron el paquete, quiero un supervisor",
                "Esta tienda me falla siempre",
                "Necesito un humano, el bot no sirve",
            ],
            "policy": [
                "Dame 40% de descuento o me voy",
                "¿Igualan un precio de la competencia bajo costo?",
                "Quiero envío gratis en un artículo de $5",
                "Reembólsame fuera de la ventana de devolución",
                "Aplica un cupón que venció el año pasado",
                "Envío internacional gratis hoy",
                "Salten la política de devoluciones por mí",
                "Exijo un descuento VIP permanente",
            ],
            "abuse": [
                "Son unos idiotas, arreglen mi pedido ya",
                "Esta tienda es una estafa, váyanse al diablo",
                "Soporte basura e inútil",
                "Cállense y denme el dinero",
                "Odio esta tienda y a todos ustedes",
                "Idiotas perdieron mi paquete",
                "Dejen de perder el tiempo bots",
                "Los voy a demandar idiotas",
            ],
        }
        reply = "Te conecto con un agente humano."

    return {
        "instruction": INSTRUCTION,
        "input": rng.choice(texts[kind]),
        "output": dump_output("handoff_human", {"reason": kind}, [], reply),
    }


GENERATORS = {
    "search_products": make_search,
    "ask": make_ask,
    "add_to_cart": make_add_to_cart,
    "track_order": make_track_order,
    "start_return": make_start_return,
    "handoff_human": make_handoff,
}

print("Generators ready:", list(GENERATORS))

In [ ]:
def validate_entry(entry: dict) -> None:
    """Raise if an instruction row or its JSON output is malformed."""
    for key in ("instruction", "input", "output"):
        if key not in entry or not isinstance(entry[key], str):
            raise ValueError(f"missing or non-string field: {key}")
    if not entry["instruction"] or not entry["input"] or not entry["output"]:
        raise ValueError("empty instruction/input/output")

    payload = json.loads(entry["output"])
    if list(payload.keys()) != ["action", "slots", "missing", "reply"]:
        raise ValueError(f"bad key order/set: {list(payload.keys())}")
    if payload["action"] not in ACTIONS:
        raise ValueError(f"unknown action: {payload['action']}")
    if not isinstance(payload["slots"], dict):
        raise ValueError("slots must be a dict")
    if not isinstance(payload["missing"], list):
        raise ValueError("missing must be a list")
    if not isinstance(payload["reply"], str) or not payload["reply"]:
        raise ValueError("reply must be a non-empty string")


def build_dataset(
    n_per_action_lang: int = 80,
    seed: int = 42,
) -> tuple[list[dict], Counter]:
    """Build a balanced ES/EN shop-assistant instruction dataset."""
    rng = random.Random(seed)
    data: list[dict] = []
    lang_tags: list[str] = []
    for action, gen in GENERATORS.items():
        for lang in ("en", "es"):
            for _ in range(n_per_action_lang):
                entry = gen(rng, lang)
                validate_entry(entry)
                data.append(entry)
                lang_tags.append(lang)
    paired = list(zip(data, lang_tags))
    rng.shuffle(paired)
    data_out, tags_out = zip(*paired)
    return list(data_out), Counter(tags_out)


data, lang_counts = build_dataset()
print("Number of entries:", len(data))
print("Languages:", dict(sorted(lang_counts.items())))
print("Example entry:\n", data[0])

In [ ]:
# Works whether the kernel cwd is the repo root or notebooks/.
cwd = Path.cwd()
DATA_DIR = cwd / "data" if (cwd / "notebooks").is_dir() else cwd.parent / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
file_path = DATA_DIR / "instruction-data.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(data)} entries to {file_path.resolve()}")

# Reload the file we just wrote.
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print("Reloaded entries:", len(data))

In [ ]:
action_counts = Counter(
    json.loads(entry["output"])["action"] for entry in data
)

print("Actions:")
for action, count in sorted(action_counts.items()):
    print(f"  {action}: {count}")
print("Languages (from generation):")
for lang, count in sorted(lang_counts.items()):
    print(f"  {lang}: {count}")

### Prompt formatting

`format_input` builds the prompt. The response section is appended at training time.

In [ ]:
def format_input(entry: dict) -> str:
    """Format an instruction entry as the training prompt."""
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text


model_input = format_input(data[0])
desired_response = f"\n\n### Response:\n{data[0]['output']}"
print(model_input + desired_response)

### Train / validation / test split

85% train, 10% test, 5% validation. The dataset was shuffled with a fixed seed so each split sees all actions.

In [ ]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion : train_portion + test_portion]
val_data = data[train_portion + test_portion :]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))


def action_counter(split: list[dict]) -> Counter:
    """Count actions in a split."""
    return Counter(json.loads(e["output"])["action"] for e in split)


print("\nActions in train:", dict(sorted(action_counter(train_data).items())))
print("Actions in val:  ", dict(sorted(action_counter(val_data).items())))
print("Actions in test: ", dict(sorted(action_counter(test_data).items())))